In [92]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [93]:
nav = pd.read_csv("data/raw/02_nav_history.csv")

transactions = pd.read_csv(
    "data/raw/08_investor_transactions.csv"
)

performance = pd.read_csv(
    "data/raw/07_scheme_performance.csv"
)

fund_master = pd.read_csv(
    "data/raw/01_fund_master.csv"
)

aum_fund = pd.read_csv(
    "data/raw/03_fund_aum.csv"
)    

In [94]:
print(nav.shape)
print(nav.head())

print(nav.info())

(46000, 3)
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692
<class 'pandas.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  str    
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), str(1)
memory usage: 1.1 MB
None


In [95]:
nav["date"] = pd.to_datetime(
    nav["date"],
    errors="coerce"
)

In [96]:
nav["date"].dtype

dtype('<M8[us]')

In [97]:
nav = nav.sort_values(
    ["amfi_code","date"]
)

In [98]:
nav["nav"] = (
    nav
    .groupby("amfi_code")["nav"]
    .ffill()
)

In [99]:
nav = nav.drop_duplicates()

In [100]:
nav = nav.drop_duplicates(
    subset=["amfi_code","date"]
)

In [101]:
invalid_nav = nav[nav["nav"]<=0]

In [102]:
nav = nav[nav["nav"]>0]

In [103]:
nav.to_csv(
    "data/processed/nav_history.csv",
    index=False
)

In [104]:
transactions["transaction_type"] = (
transactions["transaction_type"]
.str.strip()
.str.lower()
)

In [105]:
mapping = {

"sip":"SIP",

"systematic investment":"SIP",

"lumpsum":"Lumpsum",

"lump sum":"Lumpsum",

"redemption":"Redemption"

}

In [106]:
transactions["transaction_type"] = (
transactions["transaction_type"]
.replace(mapping)
)

In [107]:
transactions = transactions[
transactions["amount_inr"]>0
]

In [108]:
transactions["transaction_date"] = pd.to_datetime(

transactions["transaction_date"],

errors="coerce"
)

In [109]:
valid = [
"Verified",
"Pending",
"Rejected"
]

invalid = transactions[
~transactions["kyc_status"].isin(valid)
]

In [110]:
transactions.to_csv(
"data/processed/investor_transactions.csv",
index=False
)

In [111]:
cols = [
"return_1yr_pct",
"return_3yr_pct",
"return_5yr_pct"
]

for c in cols:

    performance[c]=pd.to_numeric(
        performance[c],
        errors="coerce"
    )

In [112]:
performance[
performance["return_1yr_pct"]>200
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [113]:
performance[
performance["return_1yr_pct"]<-100
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [114]:
performance[
(performance["expense_ratio_pct"]<0.1)
|
(performance["expense_ratio_pct"]>2.5)
]

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade


In [115]:
performance.to_csv(
"data/processed/scheme_performance.csv",
index=False
)

In [116]:
import pandas as pd

nav = pd.read_csv("data/processed/nav_history.csv")

nav["date"] = pd.to_datetime(nav["date"])

In [117]:
dim_date = pd.DataFrame()

dim_date["date"] = pd.to_datetime(nav["date"].drop_duplicates())

dim_date = dim_date.sort_values("date")

dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)

dim_date["day"] = dim_date["date"].dt.day

dim_date["month"] = dim_date["date"].dt.month

dim_date["quarter"] = dim_date["date"].dt.quarter

dim_date["year"] = dim_date["date"].dt.year
print(dim_date)

           date  date_key  day  month  quarter  year
0    2022-01-03  20220103    3      1        1  2022
1    2022-01-04  20220104    4      1        1  2022
2    2022-01-05  20220105    5      1        1  2022
3    2022-01-06  20220106    6      1        1  2022
4    2022-01-07  20220107    7      1        1  2022
...         ...       ...  ...    ...      ...   ...
1145 2026-05-25  20260525   25      5        2  2026
1146 2026-05-26  20260526   26      5        2  2026
1147 2026-05-27  20260527   27      5        2  2026
1148 2026-05-28  20260528   28      5        2  2026
1149 2026-05-29  20260529   29      5        2  2026

[1150 rows x 6 columns]


In [131]:
dim_date.to_csv(
"data/processed/dim_date_table.csv",
index=False
)

In [132]:
import sqlite3

conn = sqlite3.connect("database/bluestock_mf.db")
print("Database created successfully!")

conn.close()

Database created successfully!


In [133]:
import sqlite3

conn = sqlite3.connect("database/bluestock_mf.db")

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

print(cursor.fetchall())

conn.close()

[('dim_date',), ('fact_aum',), ('fact_nav',), ('fact_transactions',), ('fact_performance',), ('dim_fund',)]


In [134]:
engine = create_engine("sqlite:///database/bluestock_mf.db")

In [141]:
nav.to_sql(
"fact_nav",
engine,
if_exists="replace",
index=False
)

transactions.to_sql(
"fact_transactions",
engine,
if_exists="replace",
index=False
)

performance.to_sql(
"fact_performance",
engine,
if_exists="replace",
index=False
)

fund_master.to_sql(
"dim_fund",
engine,
if_exists="replace",
index=False
)

dim_date.to_sql(
"dim_date",
engine,
if_exists="replace",
index=False
)

aum_fund.to_sql(
"fact_aum",
engine,
if_exists="replace",
index=False
)

90

In [83]:
print(len(nav))

46000
